# Exercises XP — RAG with LangChain (solution complète)

**Objectif :** construire, comprendre et diagnostiquer un système simple de
**Retrieval-Augmented Generation (RAG)** sans clé API.

Le notebook utilise :

- `m-ric/huggingface_doc` comme base de connaissances ;
- `sentence-transformers/all-MiniLM-L6-v2` pour les embeddings ;
- FAISS comme base vectorielle locale ;
- `google/flan-t5-small` comme modèle de génération local ;
- LangChain pour orchestrer le découpage, la recherche et la génération.

> Le corpus est principalement en anglais. Les questions de démonstration sont
> donc formulées en anglais afin d'évaluer le système dans la langue du corpus.

## 0. Workflow général

Un système RAG sépare le travail en deux phases.

### Phase A — Indexation

1. Charger les documents.
2. Nettoyer et normaliser les textes.
3. Découper les documents en *chunks*.
4. Transformer chaque chunk en vecteur numérique (*embedding*).
5. Stocker les vecteurs et leurs métadonnées dans FAISS.

### Phase B — Question-réponse

1. Recevoir une question.
2. Transformer la question en embedding.
3. Rechercher les chunks les plus proches.
4. Insérer ces chunks dans le prompt du modèle.
5. Générer une réponse fondée sur le contexte récupéré.
6. Afficher les sources afin de vérifier la réponse.

```text
Documents → Chunks → Embeddings → FAISS
                                      ↑
Question → Embedding → Retriever ─────┘
                           ↓
                    Contexte pertinent
                           ↓
                    LLM local → Réponse
```

## 1. Installation

Les versions sont limitées à une même génération majeure afin de réduire le
risque de rupture d'API. `RetrievalQA` se trouve désormais dans
`langchain-classic`; il est utilisé ici parce que l'exercice le demande.

In [ ]:
%pip install -q \
    "datasets>=4.0,<5.0" \
    "transformers>=5.0,<6.0" \
    "sentence-transformers>=5.2,<6.0" \
    "faiss-cpu>=1.11,<2.0" \
    "langchain-core>=1.0,<2.0" \
    "langchain-classic>=1.0,<2.0" \
    "langchain-community>=0.4,<0.5" \
    "langchain-text-splitters>=1.0,<2.0" \
    "langchain-huggingface>=1.0,<2.0" \
    "accelerate>=1.8,<2.0" \
    "sentencepiece>=0.2,<1.0"

### Rôle des bibliothèques

| Bibliothèque | Rôle |
|---|---|
| `datasets` | Télécharger et lire le corpus |
| `langchain-core` | Fournir `Document` et les prompts |
| `langchain-text-splitters` | Découper les textes |
| `sentence-transformers` | Calculer les embeddings |
| `faiss-cpu` | Effectuer la recherche vectorielle |
| `langchain-community` | Fournir l'intégration FAISS |
| `langchain-huggingface` | Intégrer les modèles Hugging Face |
| `langchain-classic` | Fournir la chaîne `RetrievalQA` |
| `transformers` | Exécuter FLAN-T5 localement |

In [ ]:
import importlib.metadata as metadata
import re
import warnings
from time import perf_counter
from typing import Dict, List

import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

warnings.filterwarnings("ignore", category=FutureWarning)

packages_to_check = [
    "datasets",
    "transformers",
    "sentence-transformers",
    "faiss-cpu",
    "langchain-core",
    "langchain-classic",
    "langchain-community",
    "langchain-text-splitters",
    "langchain-huggingface",
]

print("Versions installées :")
for package_name in packages_to_check:
    try:
        print(f"- {package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"- {package_name}: non trouvé")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nMatériel utilisé : {DEVICE}")

## 2. Configuration reproductible

Les paramètres importants sont centralisés afin que l'expérience puisse être
reproduite et modifiée facilement.

- `train[:200]` limite le corpus aux 200 premières lignes.
- Deux stratégies de chunking seront comparées.
- La stratégie `balanced` sera utilisée par défaut pour la génération.

In [ ]:
DATASET_NAME = "m-ric/huggingface_doc"
DATASET_SPLIT = "train[:200]"
TEXT_COLUMN = "text"
SOURCE_COLUMN = "source"

EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_ID = "google/flan-t5-small"

CHUNKING_CONFIGS = {
    "balanced": {
        "chunk_size": 500,
        "chunk_overlap": 80,
    },
    "wide_context": {
        "chunk_size": 900,
        "chunk_overlap": 150,
    },
}

DEFAULT_STRATEGY = "balanced"
DEFAULT_K = 4

print("Configuration prête.")

## 3. Chargement et inspection du dataset

Avant tout traitement, on vérifie :

1. les colonnes disponibles ;
2. le nombre de lignes ;
3. un exemple brut ;
4. la présence des champs attendus.

Le premier chargement respecte l'énoncé. Un fallback vers le CSV officiel est
prévu si la détection automatique du dataset échoue.

In [ ]:
RAW_CSV_URL = (
    "https://huggingface.co/datasets/"
    "m-ric/huggingface_doc/resolve/main/huggingface_doc.csv"
)

try:
    ds = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
    loading_method = f"Hub dataset: {DATASET_NAME}"
except Exception as error:
    print("Chargement automatique impossible :", repr(error))
    print("Utilisation du fichier CSV officiel comme fallback.")
    ds = load_dataset(
        "csv",
        data_files=RAW_CSV_URL,
        split=DATASET_SPLIT,
    )
    loading_method = "Official CSV fallback"

print("Méthode de chargement :", loading_method)
print("Nombre de lignes :", len(ds))
print("Colonnes :", ds.column_names)

assert TEXT_COLUMN in ds.column_names, (
    f"La colonne attendue '{TEXT_COLUMN}' est absente."
)
assert SOURCE_COLUMN in ds.column_names, (
    f"La colonne attendue '{SOURCE_COLUMN}' est absente."
)

print("\nExemple brut :")
example = ds[0]
print("Source :", example[SOURCE_COLUMN])
print("Texte  :", str(example[TEXT_COLUMN])[:700])

## 4. Conversion en objets LangChain `Document`

Un objet `Document` contient :

- `page_content` : le texte à découper puis vectoriser ;
- `metadata` : les informations de traçabilité.

La source est nécessaire pour les citations. `row_id` permet de retrouver la
ligne d'origine dans le dataset.

In [ ]:
documents: List[Document] = []
skipped_rows = 0

for row_id, row in enumerate(ds):
    text = str(row.get(TEXT_COLUMN, "") or "").strip()
    source = str(row.get(SOURCE_COLUMN, "") or "").strip()

    if not text:
        skipped_rows += 1
        continue

    if not source:
        source = f"{DATASET_NAME}#row-{row_id}"

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": source,
                "row_id": row_id,
                "dataset": DATASET_NAME,
            },
        )
    )

print("Documents créés :", len(documents))
print("Lignes ignorées :", skipped_rows)
print("Métadonnées du premier document :", documents[0].metadata)
print("Extrait du premier document :")
print(documents[0].page_content[:500])

assert documents, "Aucun document exploitable n'a été créé."

### Analyse de la longueur des documents

Les documents ont des tailles très différentes. Un document très long ne doit
ni être envoyé intégralement au petit LLM, ni être représenté par un seul
embedding. Cette observation justifie le chunking.

In [ ]:
document_lengths = [len(document.page_content) for document in documents]

length_stats = pd.Series(document_lengths).describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
).to_frame("characters")

display(length_stats.round(2))
print("Document le plus long :", max(document_lengths), "caractères")

## 5. Chunking

### Pourquoi découper ?

Si un bloc mélange trop de sujets, son embedding devient peu précis. À
l'inverse, des chunks trop courts peuvent perdre le contexte.

### Paramètres

- `chunk_size` : taille maximale du chunk, en caractères ;
- `chunk_overlap` : texte partagé entre deux chunks consécutifs ;
- `add_start_index=True` : position du chunk dans le document d'origine.

L'overlap réduit le risque de couper une information importante à une frontière.

In [ ]:
def create_chunks(
    source_documents: List[Document],
    strategy_name: str,
    chunk_size: int,
    chunk_overlap: int,
) -> List[Document]:
    # Découpe les documents et enrichit les métadonnées des chunks.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_documents(source_documents)

    for chunk_id, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = chunk_id
        chunk.metadata["chunking_strategy"] = strategy_name
        chunk.metadata["chunk_size_setting"] = chunk_size
        chunk.metadata["chunk_overlap_setting"] = chunk_overlap

    return chunks


chunks_by_strategy: Dict[str, List[Document]] = {}

for strategy_name, config in CHUNKING_CONFIGS.items():
    chunks_by_strategy[strategy_name] = create_chunks(
        source_documents=documents,
        strategy_name=strategy_name,
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )

chunk_summary = []

for strategy_name, chunks in chunks_by_strategy.items():
    lengths = [len(chunk.page_content) for chunk in chunks]
    chunk_summary.append(
        {
            "strategy": strategy_name,
            "chunk_size": CHUNKING_CONFIGS[strategy_name]["chunk_size"],
            "chunk_overlap": CHUNKING_CONFIGS[strategy_name]["chunk_overlap"],
            "number_of_chunks": len(chunks),
            "average_chunk_length": round(sum(lengths) / len(lengths), 2),
            "max_chunk_length": max(lengths),
        }
    )

display(pd.DataFrame(chunk_summary))

### Inspection des chunks

Avant de calculer les embeddings, on vérifie que le texte est lisible, que la
source est conservée, que la position d'origine est présente et qu'aucun chunk
n'est vide.

In [ ]:
for strategy_name, chunks in chunks_by_strategy.items():
    print("=" * 90)
    print("STRATÉGIE :", strategy_name)
    print("Nombre de chunks :", len(chunks))
    print("Métadonnées :", chunks[0].metadata)
    print("Contenu :")
    print(chunks[0].page_content[:700])

    assert all(chunk.page_content.strip() for chunk in chunks), (
        f"Un chunk vide a été trouvé dans la stratégie {strategy_name}."
    )

## 6. Embeddings, FAISS et retriever

### Embedding

`all-MiniLM-L6-v2` transforme chaque texte en vecteur dense. Des textes
sémantiquement proches doivent produire des vecteurs proches.

### Similarité cosinus

\[
\cos(\theta)=\frac{A\cdot B}{\|A\|\|B\|}
\]

Les embeddings sont normalisés. FAISS indexe ensuite les vecteurs pour accélérer
la recherche.

### Retriever

Le retriever ne rédige pas la réponse. Il sélectionne les chunks qui formeront
le contexte fourni au LLM.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={"device": DEVICE},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,
    },
)

vectorstores: Dict[str, FAISS] = {}

for strategy_name, chunks in chunks_by_strategy.items():
    start = perf_counter()

    vectorstores[strategy_name] = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings,
        distance_strategy=DistanceStrategy.COSINE,
    )

    elapsed = perf_counter() - start
    print(
        f"Index '{strategy_name}' construit avec {len(chunks)} chunks "
        f"en {elapsed:.2f} secondes."
    )

print("Tous les index FAISS sont prêts.")

## 7. Inspection du retrieval

Une bonne pratique consiste à examiner la recherche avant de connecter le LLM.
Sinon, une erreur du retriever peut être attribuée à tort au générateur.

In [ ]:
def retrieve_documents(
    question: str,
    strategy_name: str = DEFAULT_STRATEGY,
    k: int = DEFAULT_K,
) -> List[Document]:
    # Récupère les k chunks les plus proches pour une question.
    if strategy_name not in vectorstores:
        raise ValueError(
            f"Stratégie inconnue : {strategy_name}. "
            f"Valeurs possibles : {list(vectorstores)}"
        )

    retriever = vectorstores[strategy_name].as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )
    return retriever.invoke(question)


def retrieval_to_dataframe(
    retrieved_documents: List[Document],
) -> pd.DataFrame:
    # Transforme les résultats en tableau lisible.
    records = []

    for rank, document in enumerate(retrieved_documents, start=1):
        clean_preview = re.sub(r"\s+", " ", document.page_content).strip()
        records.append(
            {
                "rank": rank,
                "source": document.metadata.get("source"),
                "row_id": document.metadata.get("row_id"),
                "chunk_id": document.metadata.get("chunk_id"),
                "start_index": document.metadata.get("start_index"),
                "characters": len(document.page_content),
                "preview": clean_preview[:420],
            }
        )

    return pd.DataFrame(records)

## 8. Comparaison des stratégies de chunking

La question et `k=4` restent constants. La seule variable modifiée est la
stratégie de découpage.

À observer :

- les sources changent-elles ?
- les chunks compacts sont-ils plus précis ?
- les chunks larges apportent-ils plus de contexte ?
- y a-t-il de la redondance ?

In [ ]:
RETRIEVAL_QUESTION = "How can I retrieve a model from the Hugging Face Hub?"

for strategy_name in CHUNKING_CONFIGS:
    print("\n" + "=" * 100)
    print(
        f"QUESTION: {RETRIEVAL_QUESTION}\n"
        f"STRATÉGIE: {strategy_name} | k={DEFAULT_K}"
    )

    retrieved = retrieve_documents(
        question=RETRIEVAL_QUESTION,
        strategy_name=strategy_name,
        k=DEFAULT_K,
    )
    display(retrieval_to_dataframe(retrieved))

## 9. Comparaison de `k = 2`, `k = 4` et `k = 6`

`k` est le nombre de chunks renvoyés.

- petit `k` : contexte ciblé, mais couverture réduite ;
- grand `k` : couverture plus large, mais davantage de bruit ;
- le meilleur `k` dépend du corpus, de la question et de la fenêtre de contexte.

In [ ]:
for k_value in [2, 4, 6]:
    print("\n" + "=" * 100)
    print(
        f"QUESTION: {RETRIEVAL_QUESTION}\n"
        f"STRATÉGIE: {DEFAULT_STRATEGY} | k={k_value}"
    )

    retrieved = retrieve_documents(
        question=RETRIEVAL_QUESTION,
        strategy_name=DEFAULT_STRATEGY,
        k=k_value,
    )
    display(retrieval_to_dataframe(retrieved))

## 10. Sanity check obligatoire

Lire attentivement les quatre chunks récupérés.

### Checklist

1. Au moins un chunk répond-il explicitement à la question ?
2. Les sources sont-elles liées à Transformers ou au Hub ?
3. Les résultats contiennent-ils du code ou des instructions utiles ?
4. Les chunks sont-ils complémentaires plutôt que répétitifs ?
5. Le contexte serait-il compréhensible par un petit modèle ?

Le contrôle automatique suivant recherche quelques mots-clés. Il ne remplace
pas la lecture humaine.

In [ ]:
sanity_documents = retrieve_documents(
    question=RETRIEVAL_QUESTION,
    strategy_name=DEFAULT_STRATEGY,
    k=DEFAULT_K,
)

display(retrieval_to_dataframe(sanity_documents))

expected_terms = {
    "hub",
    "model",
    "from_pretrained",
    "transformers",
    "download",
    "load",
}

print("\nIndicateur simple de présence de mots-clés :")
for rank, document in enumerate(sanity_documents, start=1):
    normalized_text = document.page_content.lower()
    matched_terms = sorted(
        term for term in expected_terms if term in normalized_text
    )
    print(f"- Chunk {rank}: {matched_terms or 'aucun mot-clé détecté'}")

print(
    "\nDécision expérimentale par défaut : "
    f"stratégie='{DEFAULT_STRATEGY}', k={DEFAULT_K}."
)
print(
    "Si les chunks sont peu pertinents, modifiez CHUNKING_CONFIGS, "
    "DEFAULT_STRATEGY ou DEFAULT_K, puis reconstruisez les index."
)

## 11. Chargement du modèle local `google/flan-t5-small`

FLAN-T5 est un modèle *sequence-to-sequence*. La tâche correspondante est
`text2text-generation`.

- `do_sample=False` rend les sorties plus déterministes ;
- `max_new_tokens` limite la réponse ;
- `truncation=True` protège contre les prompts trop longs ;
- le GPU est utilisé automatiquement s'il est disponible.

Ce petit modèle est adapté à l'apprentissage, mais ses réponses peuvent être
courtes ou imparfaites.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_ID)

pipeline_device = 0 if torch.cuda.is_available() else -1

hf_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=pipeline_device,
    max_new_tokens=128,
    do_sample=False,
    truncation=True,
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

print(
    f"Modèle '{LLM_MODEL_ID}' prêt sur "
    f"{'GPU' if pipeline_device == 0 else 'CPU'}."
)

## 12. Construction de `RetrievalQA`

La chaîne utilise le mode `stuff` :

1. le retriever récupère les chunks ;
2. les chunks sont concaténés dans `{context}` ;
3. la question est insérée dans `{question}` ;
4. le LLM génère la réponse.

Le prompt impose l'utilisation du contexte. `return_source_documents=True`
permet de récupérer les chunks qui ont servi à la génération.

In [ ]:
final_retriever = vectorstores[DEFAULT_STRATEGY].as_retriever(
    search_type="similarity",
    search_kwargs={"k": DEFAULT_K},
)

RAG_PROMPT_TEMPLATE = """
Answer the question using only the context below.
If the answer is not present in the context, say:
"I do not know based on the provided documents."

Give a concise and factual answer.

Context:
{context}

Question:
{question}

Answer:
""".strip()

rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=final_retriever,
    chain_type="stuff",
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt},
)

print("Chaîne RAG prête.")

## 13. Démonstration : LLM seul contre RAG

Le modèle seul utilise uniquement ses connaissances d'entraînement. Le RAG lui
fournit des passages du corpus au moment de la question.

In [ ]:
question = RETRIEVAL_QUESTION

no_rag_prompt = f"""
Answer the following question.
If you are not sure, say you are not sure.

Question: {question}
Answer:
""".strip()

no_rag_answer = hf_pipeline(no_rag_prompt)[0]["generated_text"]
rag_result = qa.invoke({"query": question})

print("QUESTION:")
print(question)

print("\n" + "=" * 90)
print("RÉPONSE SANS RAG:")
print(no_rag_answer)

print("\n" + "=" * 90)
print("RÉPONSE AVEC RAG:")
print(rag_result["result"])

print("\n" + "=" * 90)
print("SOURCES RÉCUPÉRÉES:")
for rank, document in enumerate(rag_result["source_documents"], start=1):
    print(
        f"{rank}. {document.metadata.get('source')} "
        f"(row={document.metadata.get('row_id')}, "
        f"start={document.metadata.get('start_index')})"
    )

## 14. Fonction réutilisable avec sources

La fonction suivante valide la question, exécute la chaîne, affiche la réponse,
les sources et un extrait de chaque chunk.

In [ ]:
def ask_rag(question: str, show_context: bool = True) -> dict:
    # Interroge la chaîne RAG et affiche réponse, sources et extraits.
    clean_question = question.strip()

    if not clean_question:
        raise ValueError("La question ne peut pas être vide.")

    result = qa.invoke({"query": clean_question})

    print("=" * 100)
    print("QUESTION")
    print(clean_question)

    print("\nRÉPONSE")
    print(result["result"])

    print("\nSOURCES")
    for rank, document in enumerate(
        result["source_documents"],
        start=1,
    ):
        source = document.metadata.get("source", "unknown")
        row_id = document.metadata.get("row_id")
        start_index = document.metadata.get("start_index")

        print(
            f"\n[{rank}] {source} "
            f"| row_id={row_id} | start_index={start_index}"
        )

        if show_context:
            preview = re.sub(
                r"\s+",
                " ",
                document.page_content,
            ).strip()
            print(preview[:600])

    return result

## 15. Plusieurs questions de test

Une réponse doit être jugée avec ses sources.

- La réponse est-elle soutenue par un chunk ?
- Le modèle reprend-il fidèlement les instructions du corpus ?
- Une source non pertinente influence-t-elle la réponse ?
- Le modèle reconnaît-il l'absence d'information ?

In [ ]:
test_questions = [
    "How can I retrieve a model from the Hugging Face Hub?",
    "How can I load a dataset with the Hugging Face Datasets library?",
    "What is the purpose of the from_pretrained method?",
]

test_results = {}

for test_question in test_questions:
    test_results[test_question] = ask_rag(
        test_question,
        show_context=True,
    )
    print("\n")

## 16. Question hors corpus

Un RAG fiable doit reconnaître qu'il ne possède pas la réponse. Ce test vérifie
le comportement du prompt face à une question étrangère au corpus.

In [ ]:
out_of_scope_question = (
    "What was the exact closing price of Apple stock on 15 July 2026?"
)

_ = ask_rag(out_of_scope_question, show_context=True)

## 17. Diagnostic académique d'un système RAG

### Cas 1 — Les chunks sont hors sujet

Le problème vient probablement du retrieval :

- revoir le nettoyage ;
- modifier `chunk_size`, `chunk_overlap` ou `k` ;
- vérifier la langue des requêtes ;
- tester un autre modèle d'embedding ;
- ajouter des filtres sur les métadonnées.

### Cas 2 — Les chunks sont pertinents, mais la réponse est fausse

Le problème vient probablement de la génération :

- renforcer le prompt ;
- réduire le contexte inutile ;
- utiliser un meilleur modèle ;
- clarifier la question ;
- contrôler la longueur maximale de l'entrée.

### Cas 3 — La réponse est correcte, mais les sources sont incorrectes

Vérifier la propagation de `source`, `row_id` et `start_index`, ainsi que les
doublons dans le corpus.

### Cas 4 — Le système est lent

Mesurer séparément le téléchargement, le chunking, les embeddings, la
construction de l'index, la recherche et la génération.

En production, l'indexation se fait généralement hors ligne et l'index est
sauvegardé, afin de ne pas recalculer tous les embeddings à chaque requête.

## 18. Limites

1. Le corpus est limité aux 200 premières lignes.
2. FAISS fonctionne ici en mémoire.
3. FLAN-T5-small a une capacité limitée.
4. La similarité sémantique ne garantit pas la vérité.
5. Le prompt réduit les hallucinations sans les supprimer.
6. L'évaluation est principalement qualitative.
7. `RetrievalQA` est une API historique utilisée pour respecter l'exercice.

Pour aller plus loin : jeu d'évaluation, Recall@k, MRR, évaluation de fidélité,
reranker, persistance de l'index et journalisation des sources.

## 19. Conclusion

Ce notebook montre que :

- le vector store conserve embeddings, textes et métadonnées ;
- le retriever sélectionne le contexte avant la génération ;
- le chunking influence précision et couverture ;
- `k` contrôle le compromis entre information utile et bruit ;
- le retrieval doit être vérifié avant l'appel au LLM ;
- les sources sont indispensables pour auditer la réponse.

> Une réponse RAG n'est fiable que si le corpus est pertinent, le retrieval est
> correct, le contexte est lisible et le modèle reste fidèle aux passages
> récupérés.

## 20. Références techniques

- Dataset : https://huggingface.co/datasets/m-ric/huggingface_doc
- Embeddings : https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- Modèle : https://huggingface.co/google/flan-t5-small
- Datasets : https://huggingface.co/docs/datasets/
- Pipelines : https://huggingface.co/docs/transformers/main_classes/pipelines
- FAISS LangChain : https://docs.langchain.com/oss/python/integrations/vectorstores/faiss
- LangChain Classic : https://reference.langchain.com/python/langchain_classic/